# Metabolomics overview plot w/ Sirius Canopus classifications

Date created: 12/28/2024

Notebook author: Yang Chen

Data analysis by: Britta De Pessemier

This notebook plots the following:
- Metabolomics overview plots showing number of total metabolites, with and without suspects library, and classified annotations w/ Sirius Canopus classifications

In [517]:
# Import Python packages
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from matplotlib_venn import venn3
import plotly.graph_objects as go
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib import colormaps
from upsetplot import from_indicators, plot
from upsetplot import from_memberships, UpSet

In [518]:
# Read in Sirius Canopus classifications file
sirius_result = pd.read_csv('../Data/metabolomics/Run3_10252024/canopus_structure_summary.csv')
sirius_result

,formulaRank,molecularFormula,adduct,precursorFormula,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,...,ClassyFire#most specific class,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,featureId,overallFeatureQuality
0,1,C6H15N5O3,[M + K]+,C6H15KN5O3+,Carbohydrates,0.897,Small peptides,0.649,Aminoacids,0.364,...,Alpha amino acids and derivatives,0.532,Organic compounds; Organoheterocyclic compound...,244.079,30,0.501,638448045036744826,637772591529933629,168,NaN
1,1,C4H6N3O2,[M + Na]+,C4H6N3NaO2+,Alkaloids,0.852,Pseudoalkaloids (transamidation),0.286,Purine alkaloids,0.372,...,Azoles,0.534,Organic compounds; Organoheterocyclic compound...,151.035,30,0.501,638448053119168779,637772593396399054,264,NaN
2,1,C6H15NO3,[M + H]+,C6H16NO3+,Carbohydrates,0.504,Aminosugars and aminoglycosides,0.372,Aminosugars,0.334,...,"1,2-aminoalcohols",0.896,Organic compounds; Alcohols and polyols; Organ...,150.112,31,0.509,638448044789280881,637772593182489535,261,NaN
3,19,C8H16N6O8,[M - H2O + H]+,C8H15N6O7+,Carbohydrates,0.996,Aminosugars and aminoglycosides,0.537,Aminosugars,0.198,...,Pentoses,0.533,Organic compounds; Organoheterocyclic compound...,307.102,31,0.511,638448046974513380,637772589617330870,37,NaN
4,1,C5H11N3O2,[M + H]+,C5H12N3O2+,Amino acids and Peptides,0.774,Small peptides,0.805,Aminoacids,0.910,...,Gamma amino acids and derivatives,0.844,Organic compounds; Lipids and lipid-like molec...,146.092,31,0.511,638448093829083581,637772594541444126,313,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5620,2,C30H44O,[M + K]+,C30H44KO+,Shikimates and Phenylpropanoids,0.462,Flavonoids,0.266,Chalcones,0.230,...,Phenylpropanes,0.887,Organic compounds; Ketones; Organooxygen compo...,459.301,506,8.431,638475783681175681,637772836619926454,32963,NaN
5621,1,C30H38N2O2,[M + H3N + H]+,C30H42N3O2+,Alkaloids,0.252,Anthranilic acid alkaloids,0.096,Acridone alkaloids,0.162,...,Phenylpropanes,0.965,Organic compounds; Organic acids and derivativ...,476.328,506,8.434,638475786025792128,637772836590566321,32962,NaN
5622,2,C31H49NO,[M + K]+,C31H49KNO+,Alkaloids,0.526,Pseudoalkaloids (transamidation),0.416,Phenylalanine-derived alkaloids,0.185,...,Phenylpropanes,0.783,Organic compounds; Organonitrogen compounds; O...,490.344,506,8.434,638475808943472943,637772836645092283,32964,NaN
5623,1,C28H57N3O7,[M + H]+,C28H58N3O7+,Amino acids and Peptides,0.660,Small peptides,0.182,Aminoacids,0.123,...,Amino acids and derivatives,0.563,"Organic compounds; Amino acids, peptides, and ...",548.425,507,8.453,638475808628900005,637772836926110696,33038,NaN


In [519]:
# Take a look at all the column names
sirius_result.columns

Index(['formulaRank', 'molecularFormula', 'adduct', 'precursorFormula',
       'NPC#pathway', 'NPC#pathway Probability', 'NPC#superclass',
       'NPC#superclass Probability', 'NPC#class', 'NPC#class Probability',
       'ClassyFire#superclass', 'ClassyFire#superclass probability',
       'ClassyFire#class', 'ClassyFire#class Probability',
       'ClassyFire#subclass', 'ClassyFire#subclass Probability',
       'ClassyFire#level 5', 'ClassyFire#level 5 Probability',
       'ClassyFire#most specific class',
       'ClassyFire#most specific class Probability',
       'ClassyFire#all classifications', 'ionMass', 'retentionTimeInSeconds',
       'retentionTimeInMinutes', 'formulaId', 'alignedFeatureId', 'featureId',
       'overallFeatureQuality'],
      dtype='object')

In [520]:
# Check that the number of features is both unique and equal to the number of rows in the DataFrame
if sirius_result['featureId'].nunique() == sirius_result.shape[0]:
    print(f"There are {sirius_result['featureId'].nunique()} different metabolites.")
else:
    print("The number of unique features does not match the number of rows in the Sirius result.")

There are 5625 different metabolites.


In [521]:
### Check that the n=5625 metabolites from Sirius are all within the total n=7683 metabolites

# Read the Sirius features into a set
sirius_features = set(sirius_result['featureId'])

# Read the info_feature_complete dataset
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')

# Total number of metabolites in info_feature_complete
total_num_metabolites = info_feature_complete.shape[0]

# Check if all Sirius features are in the complete set
sirius_in_total = sirius_features.issubset(info_feature_complete['Feature'])

# Display results
if sirius_in_total:
    print(f"All {len(sirius_features)} metabolites from Sirius are within the total of {total_num_metabolites} metabolites.")
else:
    # Find how many Sirius features are missing
    missing_features = sirius_features - set(info_feature_complete['featureId'])
    print(f"{len(missing_features)} metabolites from Sirius are not in the total of {total_num_metabolites} metabolites.")


All 5625 metabolites from Sirius are within the total of 7683 metabolites.


In [522]:
### Check how many of the n=1027 metabolites from GNPS suspect spectral library are within the Sirius result of n=5625

# Expecting most of them to be here...though it's possible Sirius was able to give decently confident predictions that GNPS did not at all as they rely on separate spectral databases

# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')

# Find the intersection between the two columns
matching_scans = gnps_with_suspects['#Scan#'].isin(sirius_result['featureId'])

# Count the number of matches
count = matching_scans.sum()

# Calculate the percentage of matches
percentage = (count / gnps_with_suspects.shape[0]) * 100

# Display the result
print(f"Number of #Scan# values in gnps_with_suspects that are within sirius_result['featureId']: {count}, {percentage:.2f}%")

Number of #Scan# values in gnps_with_suspects that are within sirius_result['featureId']: 918, 89.39%


In [523]:
# Set probability threshold
# (reccomendation of 0.75 from the Dorrestein lab)
prob_thres = 0.75

In [524]:
### Check probability calls at various levels for NPC designations

# Count the total number of rows
total_rows = sirius_result.shape[0]

# Define the columns to check
columns = ['NPC#pathway Probability', 'NPC#superclass Probability', 'NPC#class Probability']

# Iterate through the columns, calculate counts and percentages, and display results
for col in columns:
    count = sirius_result[sirius_result[col] >= prob_thres].shape[0]
    percentage = (count / total_rows) * 100
    print(f"Number of rows with {col} >= {prob_thres}: {count}, {percentage:.2f}%")


Number of rows with NPC#pathway Probability >= 0.75: 3222, 57.28%
Number of rows with NPC#superclass Probability >= 0.75: 1982, 35.24%
Number of rows with NPC#class Probability >= 0.75: 1363, 24.23%


In [525]:
### Check probability calls at various levels for ClassyFire designations

# Count the total number of rows
total_rows = sirius_result.shape[0]

# Define the columns to check for ClassyFire designations
columns = [
    'ClassyFire#superclass probability', 
    'ClassyFire#class Probability', 
    'ClassyFire#subclass Probability', 
    'ClassyFire#level 5 Probability', 
    'ClassyFire#most specific class Probability'
]

# Iterate through the columns, calculate counts and percentages, and display results
for col in columns:
    count = sirius_result[sirius_result[col] >= prob_thres].shape[0]
    percentage = (count / total_rows) * 100
    print(f"Number of rows with {col} >= {prob_thres}: {count}, {percentage:.2f}%")


Number of rows with ClassyFire#superclass probability >= 0.75: 4519, 80.34%
Number of rows with ClassyFire#class Probability >= 0.75: 3857, 68.57%
Number of rows with ClassyFire#subclass Probability >= 0.75: 2233, 39.70%
Number of rows with ClassyFire#level 5 Probability >= 0.75: 1379, 24.52%
Number of rows with ClassyFire#most specific class Probability >= 0.75: 1855, 32.98%


In [526]:
### Check how many features have probability of >= 0.75 across ALL the classyfire designations/columns

# Filter rows where all specified columns have values >= prob_thres
filtered_df = sirius_result[(sirius_result[columns] >= prob_thres).all(axis=1)]

# Count the number of rows that meet the condition
count = filtered_df.shape[0]

# Calculate the percentage of such rows
percentage = (count / sirius_result.shape[0]) * 100

# Display the result
print(f"Number of features with probability >= {prob_thres} across all ClassyFire designations: {count}, {percentage:.2f}%")

Number of features with probability >= 0.75 across all ClassyFire designations: 968, 17.21%


In [527]:
### Check how many features have probability of >= 0.75 across the 3 main classification levels: superclass, class, subclass

# Filter rows where the first three ClassyFire columns have values >= prob_thres
filtered_df = sirius_result[
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

# Count and percentage
count = filtered_df.shape[0]
percentage = (count / sirius_result.shape[0]) * 100

# Display the result
print(f"Number of features with probability >= {prob_thres} across the first three ClassyFire designations: {count}, {percentage:.2f}%")

Number of features with probability >= 0.75 across the first three ClassyFire designations: 2136, 37.97%


In [528]:
# Count non-NaN values for each column
non_nan_counts = sirius_result[['ClassyFire#superclass', 'ClassyFire#class', 'ClassyFire#subclass']].notna().sum()

# Display results
for column, count in non_nan_counts.items():
    percentage = (count / sirius_result.shape[0]) * 100
    print(f"Number of non-NaN values in {column}: {count}, {percentage:.2f}%")

Number of non-NaN values in ClassyFire#superclass: 5625, 100.00%
Number of non-NaN values in ClassyFire#class: 5347, 95.06%
Number of non-NaN values in ClassyFire#subclass: 4126, 73.35%


In [529]:
# Filter rows where columns are not NaN and probabilities are >= prob_thres
filtered_counts = {
    'ClassyFire#superclass': sirius_result[
        sirius_result['ClassyFire#superclass'].notna() &
        (sirius_result['ClassyFire#superclass probability'] >= prob_thres)
    ].shape[0],
    'ClassyFire#class': sirius_result[
        sirius_result['ClassyFire#class'].notna() &
        (sirius_result['ClassyFire#class Probability'] >= prob_thres)
    ].shape[0],
    'ClassyFire#subclass': sirius_result[
        sirius_result['ClassyFire#subclass'].notna() &
        (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
    ].shape[0]
}

# Display results
for column, count in filtered_counts.items():
    percentage = (count / sirius_result.shape[0]) * 100
    print(f"Number of non-NaN values in {column} with probability >= {prob_thres}: {count}, {percentage:.2f}%")

Number of non-NaN values in ClassyFire#superclass with probability >= 0.75: 4519, 80.34%
Number of non-NaN values in ClassyFire#class with probability >= 0.75: 3857, 68.57%
Number of non-NaN values in ClassyFire#subclass with probability >= 0.75: 2233, 39.70%


In [530]:
# Filter rows where all classification levels are not NaN and all probabilities are >= prob_thres (must be true for ALL)
filtered_df = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

# Count the number of rows that meet the condition
count = filtered_df.shape[0]

# Calculate the percentage of such rows
percentage = (count / sirius_result.shape[0]) * 100

# Display the result
print(f"Number of rows with all classification levels not NaN and probabilities >= {prob_thres}: {count}, {percentage:.2f}%")

Number of rows with all classification levels not NaN and probabilities >= 0.75: 2136, 37.97%


In [531]:
### Check how many of the n=1027 metabolites from GNPS suspect spectral library are within the Sirius result of n=2136

# Expecting *majority?* of them to be here...though it's possible Sirius was able to give decently confident predictions that GNPS did not at all as they rely on separate spectral databases
# Probably less than 89% though from the all Sirius comparison above

# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')

# Find the intersection between the two columns
matching_scans = gnps_with_suspects['#Scan#'].isin(filtered_df['featureId'])

# Count the number of matches
count = matching_scans.sum()

# Calculate the percentage of matches
percentage = (count / gnps_with_suspects.shape[0]) * 100

# Display the result
print(f"Number of #Scan# values in gnps_with_suspects that are within Sirius all main 3 levels not NaN and prob >= 0.75: {count}, {percentage:.2f}%")

Number of #Scan# values in gnps_with_suspects that are within Sirius all main 3 levels not NaN and prob >= 0.75: 473, 46.06%


### GNPS results

In [532]:
# Read in table of all spectra and output number of total metabolites
info_feature_complete = pd.read_csv('../Data/metabolomics/Run3_10252024/info_feature_complete.csv')
total_num_metabolites = info_feature_complete.shape[0]
print(f'Total number of consensus MS2 spectra metabolites from GNPS2: ' + str(total_num_metabolites))

Total number of consensus MS2 spectra metabolites from GNPS2: 7683


In [533]:
# Read in GNPS2 metabolites table obtained WITHOUT suspects library
gnps_without_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps_withoutsuspects.tsv', sep='\t')
num_gnps_without_suspects = gnps_without_suspects.shape[0]
print(f'Number of GNPS2 outputed metabolites WITHOUT suspects library: ' + str(num_gnps_without_suspects))

Number of GNPS2 outputed metabolites WITHOUT suspects library: 432


In [545]:
filtered_df = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

filtered_df

,formulaRank,molecularFormula,adduct,precursorFormula,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,...,ClassyFire#most specific class,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,featureId,overallFeatureQuality
2,1,C6H15NO3,[M + H]+,C6H16NO3+,Carbohydrates,0.504,Aminosugars and aminoglycosides,0.372,Aminosugars,0.334,...,"1,2-aminoalcohols",0.896,Organic compounds; Alcohols and polyols; Organ...,150.112,31,0.509,638448044789280881,637772593182489535,261,NaN
3,19,C8H16N6O8,[M - H2O + H]+,C8H15N6O7+,Carbohydrates,0.996,Aminosugars and aminoglycosides,0.537,Aminosugars,0.198,...,Pentoses,0.533,Organic compounds; Organoheterocyclic compound...,307.102,31,0.511,638448046974513380,637772589617330870,37,NaN
5,1,C8H16N4O3,[M + H]+,C8H17N4O3+,Amino acids and Peptides,0.980,Small peptides,0.997,Dipeptides,0.791,...,N-acyl-L-alpha-amino acids,0.566,Organic compounds; Lipids and lipid-like molec...,217.129,31,0.511,638448094017827270,637772603114602422,1013,NaN
8,8,C9H11N5O2,[M + H]+,C9H12N5O2+,Carbohydrates,0.662,Nucleosides,0.673,Purine alkaloids,0.720,...,6-aminopurines,0.653,Organic compounds; Organoheterocyclic compound...,222.097,31,0.513,638448142449456314,637772592301685625,216,NaN
9,1,C18H33NO15,[M + H]+,C18H34NO15+,Carbohydrates,1.000,Saccharides,0.994,Polysaccharides,0.660,...,Disaccharides,0.633,Organic compounds; Organoheterocyclic compound...,504.192,31,0.513,638448140373275749,637772603525644264,1061,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5617,1,C6H14N2,[M + H]+,C6H15N2+,Alkaloids,0.530,Ornithine alkaloids,0.258,Polyamines,0.584,...,Dialkylamines,0.670,Organic compounds; Organonitrogen compounds; D...,115.123,505,8.422,638475804086467823,637772835634264859,32752,NaN
5619,1,C6H9N3,[M + H]+,C6H10N3+,Alkaloids,0.962,Nicotinic acid alkaloids,0.176,Pyridine alkaloids,0.173,...,Aminopyrimidines and derivatives,0.855,Organic compounds; Organoheterocyclic compound...,124.087,505,8.425,638475779147132921,637772836515068834,32921,NaN
5620,2,C30H44O,[M + K]+,C30H44KO+,Shikimates and Phenylpropanoids,0.462,Flavonoids,0.266,Chalcones,0.230,...,Phenylpropanes,0.887,Organic compounds; Ketones; Organooxygen compo...,459.301,506,8.431,638475783681175681,637772836619926454,32963,NaN
5621,1,C30H38N2O2,[M + H3N + H]+,C30H42N3O2+,Alkaloids,0.252,Anthranilic acid alkaloids,0.096,Acridone alkaloids,0.162,...,Phenylpropanes,0.965,Organic compounds; Organic acids and derivativ...,476.328,506,8.434,638475786025792128,637772836590566321,32962,NaN


In [546]:
gnps_without_suspects

,SpectrumID,#Scan#,SpectrumFile,LibraryName,MQScore,TIC_Query,RT_Query,MZErrorPPM,SharedPeaks,MassDiff,...,molecular_formula,InChIKey,InChIKey-Planar,superclass,class,subclass,npclassifier_superclass,npclassifier_class,npclassifier_pathway,library_usi
0,CCMSLIB00005883855,67,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.997678,3961000.0,0,0.839914,6,0.000099,...,C5H11NO2,KZSNJWFQEVHDMF-BYPYZUCNSA-N,KZSNJWFQEVHDMF,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883855
1,CCMSLIB00012249715,2239,spectra_filtered.mgf,CMMC-REFRAME-POSITIVE-LIBRARY.mgf,0.995422,9930000.0,0,2.796090,7,0.000504,...,C6H12O6,LKDRXBCSQODPBY-UHFFFAOYSA-N,LKDRXBCSQODPBY,Organic oxygen compounds,Organooxygen compounds,Carbohydrates and carbohydrate conjugates,Saccharides,Monosaccharides,Carbohydrates,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012249715
2,CCMSLIB00012716670,23882,spectra_filtered.mgf,GNPS-N-ACYL-LIPIDS-MASSQL.mgf,0.992183,2779000.0,0,1.793690,8,0.000488,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012716670
3,CCMSLIB00005883686,1368,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.992042,12116000.0,0,2.310150,5,0.000305,...,C6H13NO2,AGPKZVBTJJNPAG-WHFBIAKZSA-N,AGPKZVBTJJNPAG,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Small peptides,Aminoacids,Amino acids and Peptides,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005883686
4,CCMSLIB00005884056,1974,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.991977,2339000.0,0,0.000000,5,0.000000,...,C6H11NO3,UZTFMUBKZQVKLK-UHFFFAOYSA-N,UZTFMUBKZQVKLK,Organic acids and derivatives,Carboxylic acids and derivatives,"Amino acids, peptides, and analogues",Fatty Acids and Conjugates,Unsaturated fatty acids,Fatty acids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00005884056
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
427,CCMSLIB00004715189,21585,spectra_filtered.mgf,MONA.mgf,0.605988,350200.0,0,2.549490,5,0.000793,...,C18H30O4,GUVJPXABQYFWPD-GSZDNMEJSA-N,GUVJPXABQYFWPD,Lipids and lipid-like molecules,Prenol lipids,Sesquiterpenoids,Diterpenoids,Labdane diterpenoids,Terpenoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00004715189
428,CCMSLIB00003136040,18597,spectra_filtered.mgf,GNPS-NIST14-MATCHES.mgf,0.602108,335300.0,0,5.583870,6,0.001190,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00003136040
429,CCMSLIB00012443965,11960,spectra_filtered.mgf,MSNLIB-POSITIVE.mgf,0.601783,1953600.0,0,2.178520,10,0.000885,...,C19H32O8,JVIDANAJZDBKRL-FJUMDGKUSA-N,JVIDANAJZDBKRL,Lipids and lipid-like molecules,Prenol lipids,Terpene glycosides,Apocarotenoids,Megastigmanes,Terpenoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00012443965
430,CCMSLIB00013558434,12746,spectra_filtered.mgf,GNPS-LIBRARY.mgf,0.601377,10408000.0,0,78.356900,5,0.019989,...,C16H14O3,RGSUZUQISVAJJF-UHFFFAOYSA-N,RGSUZUQISVAJJF,Phenylpropanoids and polyketides,Neoflavonoids,Dalbergiones,Flavonoids,Open-chained neoflavonoids,Shikimates and Phenylpropanoids,mzspec:GNPS:GNPS-LIBRARY:CCMSLIB00013558434


In [547]:


# Find the intersection between the two columns
# matching_scans = filtered_df['featureId'].isin(gnps_without_suspects['#Scan#'])
matching_scans = gnps_without_suspects['#Scan#'].isin(filtered_df['featureId'])

# Count the number of matches
count = matching_scans.sum()

# Calculate the percentage of matches
tot_annotations_sirius = filtered_df.shape[0]
percentage = (count / tot_annotations_sirius) * 100

# Display the result
print(f"Number of #Scan# values in num_gnps_without_suspects (n={gnps_without_suspects.shape[0]}) that are Sirius prob >= 0.75 predicted']: {count}, {percentage:.2f}%")

Number of #Scan# values in num_gnps_without_suspects (n=432) that are Sirius prob >= 0.75 predicted']: 235, 11.00%


In [535]:
# Read in GNPS2 metabolites table obtained WITH suspects library
gnps_with_suspects = pd.read_csv('../Data/metabolomics/Run3_10252024/merged_results_with_gnps.tsv', sep='\t')
num_gnps_with_suspects = gnps_with_suspects.shape[0]
print(f'Number of GNPS2 outputed metabolites WITH suspects library: ' + str(num_gnps_with_suspects))

Number of GNPS2 outputed metabolites WITH suspects library: 1027


In [536]:
# Count rows where 'superclass' is not NaN
num_annotated_metabolites = gnps_with_suspects['superclass'].notna().sum()

print(f"Number of superclass annotated metabolites: {num_annotated_metabolites}")

Number of superclass annotated metabolites: 250


### Nested Venn Diagram showing number of metabolites from GNPS and Sirius

In [537]:
# Filter rows where all classification levels are not NaN and probabilities are >= prob_thres (must be true for ALL)
filtered_df = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

# Define updated circle sizes
size1 = total_num_metabolites  # Size of Circle 1 (Total MS2 spectra identified)
size2 = sirius_result.shape[0]  # Size of Circle 2 (Sirius metabolites)
size3 = filtered_df.shape[0]  # Size of Circle 3 (Sirius with confident classifications)
size4 = num_gnps_with_suspects  # Size of Circle 4 (GNPS suspect spectral library matches)
size5 = num_gnps_without_suspects  # Size of Circle 5 (GNPS non-suspect spectral library matches)

# Calculate radii based on the square root of the sizes (scaled for visualization)
scaling_factor = 0.005  # Adjust this for appropriate visualization

radius1 = np.sqrt(size1) * scaling_factor  # Radius for Circle 1
radius2 = np.sqrt(size2) * scaling_factor  # Radius for Circle 2
radius3 = np.sqrt(size3) * scaling_factor  # Radius for Circle 3
radius4 = np.sqrt(size4) * scaling_factor  # Radius for Circle 4 (GNPS suspect)
radius5 = np.sqrt(size5) * scaling_factor  # Radius for Circle 5

# Calculate the percentage for each circle relative to the total size
percent2 = (size2 / size1) * 100
percent3 = (size3 / size1) * 100
percent4 = (size4 / size1) * 100
# percent5 = (size5 / size1) * 100

# Create a figure
fig, ax = plt.subplots(figsize=(6,6))

# Create circles with correct placements to represent overlap
circle1 = plt.Circle((0.5, 0.5), radius1, facecolor='#000000', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size1} total MS2 spectra identified')
circle2 = plt.Circle((0.5, 0.5), radius2, facecolor='#00305d', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size2} total annotations from Sirius4')
circle3 = plt.Circle((0.42, 0.4), radius3, facecolor='#0096FF', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size3} annotations from Sirius4 (prob >= 0.75)')
# Note that of the 1027 from GNPS Suspect Library, 89% were within the 5625 total from Sirius, and 46% were within the 2136 of Sirius prob >= 0.75

# GNPS suspect spectral library circle (slightly shifted to show partial overlap)
circle4 = plt.Circle((0.54, 0.24), radius4, facecolor='#ADD8E6', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size4} annotations from GNPS2 Suspect Spectral Library')
# circle5 = plt.Circle((0.5, 0.1), radius5, facecolor='#000080', alpha=0.6, edgecolor='black', linewidth=2, label=f'{size5} annotations from GNPS2 non-suspect spectral library')

# Add the circles to the plot
ax.add_patch(circle1)
ax.add_patch(circle2)
ax.add_patch(circle3)
ax.add_patch(circle4)
# ax.add_patch(circle5)

# Add percentage text inside the circles
ax.text(0.505, 0.68, f'{percent2:.1f}%', ha='center', va='center', fontsize=10, color='white')
ax.text(0.4105, 0.45, f'{percent3:.1f}%', ha='center', va='center', fontsize=10, color='white')
ax.text(0.54, 0.26, f'{percent4:.1f}%', ha='center', va='center', fontsize=10, color='black')
# ax.text(0.505, 0.13, f'{percent5:.1f}%', ha='center', va='center', fontsize=10, color='white')

# Set limits to ensure all circles are fully visible
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

# Set aspect ratio to be equal to ensure circles are circular
ax.set_aspect('equal')

# Remove all borders
for spine in ax.spines.values():
    spine.set_visible(False)

# Remove ticks and labels
ax.set_xticks([])
ax.set_yticks([])

# Add legend for the circles with their respective sizes and place it outside the plot
ax.legend(loc='upper left', bbox_to_anchor=(0.05, 0), fontsize=10)

# Add title with better font style and size
plt.title("Metabolites Identified from Workflow", loc='center', fontsize=16)

# Adjust layout to make room for the legend
plt.tight_layout()

# Save figure
plt.savefig('../Figures/metabolomics_Figures/overview/ms2_venn_diagram_with_sirius.png', dpi=600)

### Sankey plot of Sirius metabolites >= 75% predicted probability (n=2136)

In [538]:
# Filter rows where all classification levels are not NaN and all probabilities are >= prob_thres (must be true for ALL)
filtered_df = sirius_result[
    sirius_result['ClassyFire#superclass'].notna() &
    sirius_result['ClassyFire#class'].notna() &
    sirius_result['ClassyFire#subclass'].notna() &
    (sirius_result['ClassyFire#superclass probability'] >= prob_thres) &
    (sirius_result['ClassyFire#class Probability'] >= prob_thres) &
    (sirius_result['ClassyFire#subclass Probability'] >= prob_thres)
]

filtered_df

,formulaRank,molecularFormula,adduct,precursorFormula,NPC#pathway,NPC#pathway Probability,NPC#superclass,NPC#superclass Probability,NPC#class,NPC#class Probability,...,ClassyFire#most specific class,ClassyFire#most specific class Probability,ClassyFire#all classifications,ionMass,retentionTimeInSeconds,retentionTimeInMinutes,formulaId,alignedFeatureId,featureId,overallFeatureQuality
2,1,C6H15NO3,[M + H]+,C6H16NO3+,Carbohydrates,0.504,Aminosugars and aminoglycosides,0.372,Aminosugars,0.334,...,"1,2-aminoalcohols",0.896,Organic compounds; Alcohols and polyols; Organ...,150.112,31,0.509,638448044789280881,637772593182489535,261,NaN
3,19,C8H16N6O8,[M - H2O + H]+,C8H15N6O7+,Carbohydrates,0.996,Aminosugars and aminoglycosides,0.537,Aminosugars,0.198,...,Pentoses,0.533,Organic compounds; Organoheterocyclic compound...,307.102,31,0.511,638448046974513380,637772589617330870,37,NaN
5,1,C8H16N4O3,[M + H]+,C8H17N4O3+,Amino acids and Peptides,0.980,Small peptides,0.997,Dipeptides,0.791,...,N-acyl-L-alpha-amino acids,0.566,Organic compounds; Lipids and lipid-like molec...,217.129,31,0.511,638448094017827270,637772603114602422,1013,NaN
8,8,C9H11N5O2,[M + H]+,C9H12N5O2+,Carbohydrates,0.662,Nucleosides,0.673,Purine alkaloids,0.720,...,6-aminopurines,0.653,Organic compounds; Organoheterocyclic compound...,222.097,31,0.513,638448142449456314,637772592301685625,216,NaN
9,1,C18H33NO15,[M + H]+,C18H34NO15+,Carbohydrates,1.000,Saccharides,0.994,Polysaccharides,0.660,...,Disaccharides,0.633,Organic compounds; Organoheterocyclic compound...,504.192,31,0.513,638448140373275749,637772603525644264,1061,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5617,1,C6H14N2,[M + H]+,C6H15N2+,Alkaloids,0.530,Ornithine alkaloids,0.258,Polyamines,0.584,...,Dialkylamines,0.670,Organic compounds; Organonitrogen compounds; D...,115.123,505,8.422,638475804086467823,637772835634264859,32752,NaN
5619,1,C6H9N3,[M + H]+,C6H10N3+,Alkaloids,0.962,Nicotinic acid alkaloids,0.176,Pyridine alkaloids,0.173,...,Aminopyrimidines and derivatives,0.855,Organic compounds; Organoheterocyclic compound...,124.087,505,8.425,638475779147132921,637772836515068834,32921,NaN
5620,2,C30H44O,[M + K]+,C30H44KO+,Shikimates and Phenylpropanoids,0.462,Flavonoids,0.266,Chalcones,0.230,...,Phenylpropanes,0.887,Organic compounds; Ketones; Organooxygen compo...,459.301,506,8.431,638475783681175681,637772836619926454,32963,NaN
5621,1,C30H38N2O2,[M + H3N + H]+,C30H42N3O2+,Alkaloids,0.252,Anthranilic acid alkaloids,0.096,Acridone alkaloids,0.162,...,Phenylpropanes,0.965,Organic compounds; Organic acids and derivativ...,476.328,506,8.434,638475786025792128,637772836590566321,32962,NaN


In [539]:
filtered_df['ClassyFire#superclass'].value_counts()

ClassyFire#superclass
Organic acids and derivatives                619
Organic oxygen compounds                     613
Lipids and lipid-like molecules              345
Benzenoids                                   298
Organic nitrogen compounds                   109
Organoheterocyclic compounds                  88
Phenylpropanoids and polyketides              51
Organosulfur compounds                         6
Nucleosides, nucleotides, and analogues        2
Organic 1,3-dipolar compounds                  2
Lignans, neolignans and related compounds      2
Organometallic compounds                       1
Name: count, dtype: int64

In [540]:
filtered_df['ClassyFire#class'].value_counts()

ClassyFire#class
Organooxygen compounds                      613
Carboxylic acids and derivatives            599
Benzene and substituted derivatives         275
Fatty Acyls                                 270
Organonitrogen compounds                    109
Prenol lipids                                38
Diazines                                     34
Flavonoids                                   26
Imidazopyrimidines                           22
Phenol ethers                                16
Glycerolipids                                13
Glycerophospholipids                         12
Steroids and steroid derivatives             11
Diarylheptanoids                             10
Tannins                                       9
Azoles                                        8
Phenols                                       7
Organic phosphonic acids and derivatives      6
Thioethers                                    5
Pyridines and derivatives                     5
Benzofurans            

In [541]:
filtered_df['ClassyFire#subclass'].value_counts()

ClassyFire#subclass
Amino acids, peptides, and analogues         528
Carbohydrates and carbohydrate conjugates    331
Benzoic acids and derivatives                179
Ethers                                       134
Fatty acids and conjugates                   122
                                            ... 
Methylpyridines                                1
Pyridinecarboxylic acids and derivatives       1
Phenylphosphines and derivatives               1
N-phenylureas                                  1
Glycerophosphoserines                          1
Name: count, Length: 107, dtype: int64

In [542]:
min_count_threshold = 10

# Count occurrences for each unique label in superclass, class, and subclass
superclass_counts = filtered_df['ClassyFire#superclass'].value_counts()
class_counts = filtered_df['ClassyFire#class'].value_counts()
subclass_counts = filtered_df['ClassyFire#subclass'].value_counts()

# Filter out categories with fewer than min_count_threshold in class and subclass
filtered_df = filtered_df[
    filtered_df['ClassyFire#superclass'].isin(superclass_counts[superclass_counts >= min_count_threshold].index) &
    filtered_df['ClassyFire#class'].isin(class_counts[class_counts > min_count_threshold].index) &
    filtered_df['ClassyFire#subclass'].isin(subclass_counts[subclass_counts >= min_count_threshold].index)
]

print(f'Number of metabolites after filtering out superclass, class and subclass groups < {min_count_threshold}: {filtered_df.shape[0]}')

Number of metabolites after filtering out superclass, class and subclass groups < 10: 1935


In [543]:
# Recalculate unique labels after collapsing class and subclass
superclass_labels = filtered_df['ClassyFire#superclass'].unique()  # All categories stay in superclass
class_labels = filtered_df['ClassyFire#class'].unique()  # Collapsed classes
subclass_labels = filtered_df['ClassyFire#subclass'].unique()  # Collapsed subclasses

# Sort superclass categories by their counts in descending order
sorted_superclass_counts = superclass_counts.sort_values(ascending=False)

# Get the sorted order of superclass labels
sorted_superclass_labels = sorted_superclass_counts.index.tolist()

In [544]:
# Remove rows with NaN values in the 'class' or 'subclass' columns
filtered_df.dropna(subset=['ClassyFire#class', 'ClassyFire#subclass'], inplace=True)

# Count occurrences for each unique label in superclass, class, and subclass
superclass_counts = filtered_df['ClassyFire#superclass'].value_counts()
class_counts = filtered_df['ClassyFire#class'].value_counts()
subclass_counts = filtered_df['ClassyFire#subclass'].value_counts()

# Recalculate unique labels after collapsing class and subclass
superclass_labels = filtered_df['ClassyFire#superclass'].unique()  # All categories stay in superclass
class_labels = filtered_df['ClassyFire#class'].unique()  # Collapsed classes
subclass_labels = filtered_df['ClassyFire#subclass'].unique()  # Collapsed subclasses

# Sort superclass categories by their counts in descending order
sorted_superclass_counts = superclass_counts.sort_values(ascending=False)

# Get the sorted order of superclass labels
sorted_superclass_labels = sorted_superclass_counts.index.tolist()

# Recalculate y positions based on sorted superclass labels
superclass_y = [i / len(sorted_superclass_labels) for i in range(len(sorted_superclass_labels))]

# Assign x positions for superclass, class, and subclass categories
superclass_x = [0.3] * len(sorted_superclass_labels)  # Superclass nodes at 30% width
class_y = [i / len(class_labels) for i in range(len(class_labels))]
class_x = [0.6] * len(class_labels)  # Class nodes at 60% width
subclass_y = [i / len(subclass_labels) for i in range(len(subclass_labels))]
subclass_x = [0.9] * len(subclass_labels)  # Subclass nodes at 90% width

# Combine all positions
node_x = [0] + superclass_x + class_x + subclass_x + [1]  # Include root and unannotated node positions
node_y = [1] + superclass_y + class_y + subclass_y + [0]  # Include root and unannotated node positions

# Update the label order to match the sorted superclass labels
sorted_all_labels = ['All classified annotations\n(n=2136)'] + sorted_superclass_labels + list(class_labels) + list(subclass_labels)


# Create a dictionary to map labels to indices
label_dict = {label: idx for idx, label in enumerate(sorted_all_labels)}

# Generate source, target, and value lists for the Sankey plot
sources = []
targets = []
values = []

# 'GNPS2 identified metabolites (without suspect library)' -> Superclass
for idx, row in filtered_df.iterrows():
    source_idx = label_dict['All classified annotations\n(n=2136)']
    superclass_idx = label_dict[row['ClassyFire#superclass']]
    sources.append(source_idx)
    targets.append(superclass_idx)
    values.append(1)  # Assuming one metabolite per row, otherwise adjust the value

# Superclass -> Class
for idx, row in filtered_df.iterrows():
    superclass_idx = label_dict[row['ClassyFire#superclass']]
    class_idx = label_dict[row['ClassyFire#class']]
    sources.append(superclass_idx)
    targets.append(class_idx)
    values.append(1)  # Assuming one metabolite per row

# Class -> Subclass
for idx, row in filtered_df.iterrows():
    class_idx = label_dict[row['ClassyFire#class']]
    subclass_idx = label_dict[row['ClassyFire#subclass']]
    sources.append(class_idx)
    targets.append(subclass_idx)
    values.append(1)  # Again, assuming one metabolite per row

# Generate y positions for nodes based on their category
# superclass_y = [i / len(superclass_labels) for i in range(len(superclass_labels))]
# class_y = [i / len(class_labels) for i in range(len(class_labels))]
# subclass_y = [i / len(subclass_labels) for i in range(len(subclass_labels))]

# Define spacing to avoid overlapping nodes
superclass_y = np.linspace(0, 1, len(superclass_labels), endpoint=True)
class_y = np.linspace(0, 1, len(class_labels), endpoint=True)
subclass_y = np.linspace(0, 1, len(subclass_labels), endpoint=True)


# Generate x positions for each category of nodes
superclass_y = np.linspace(0.1, 0.9, len(superclass_x), endpoint=True).tolist()
class_y = np.linspace(0.1, 0.9, len(class_x), endpoint=True).tolist()
subclass_y = np.linspace(0.1, 0.9, len(subclass_x), endpoint=True).tolist()

# Combine all positions
node_x = [0] + superclass_x + class_x + subclass_x + [1]  # Add positions for the root and unannotated node
node_y = [0] + superclass_y + class_y + subclass_y + [5]  # Add positions for the root and unannotated node

# Define the colormap
tab10 = plt.get_cmap("tab10")

# Get the sorted superclass labels based on frequency (descending order)
sorted_superclass_labels = superclass_counts.sort_values(ascending=False).index

# Initialize the color palette list
superclass_palette = []

# Track used colors to avoid repeating orange
used_colors = set()

for i, label in enumerate(sorted_superclass_labels):
    if i == 2:  
        color = tab10(1)  # Force the third-largest category to be orange
    else:
        # Find the next available color that is NOT orange
        for j in range(10):  # Loop over tab10 indices
            if j != 1 and tab10(j / 9) not in used_colors:  # Skip orange (index 1)
                color = tab10(j / 9)
                used_colors.add(color)
                break

    superclass_palette.append(color)
    used_colors.add(color)  # Mark color as used

# Convert to hex and assign to labels
superclass_colors = {label: mcolors.to_hex(superclass_palette[i]) for i, label in enumerate(sorted_superclass_labels)}

# Generate colors for classes and subclasses based on superclass colors
node_colors = []
for label in sorted_all_labels:
    if label in superclass_labels:
        node_colors.append(superclass_colors[label])  # Use superclass color
    elif label in class_labels:
        # Derive lighter shades from superclass color
        superclass = filtered_df[filtered_df['ClassyFire#class'] == label].iloc[0]['ClassyFire#superclass']
        node_colors.append(mcolors.to_hex(mcolors.to_rgba(superclass_colors[superclass], alpha=0.8)))
    elif label in subclass_labels:
        # Derive even lighter shades from class color
        related_class = filtered_df[filtered_df['ClassyFire#subclass'] == label].iloc[0]['ClassyFire#class']
        related_superclass = filtered_df[filtered_df['ClassyFire#subclass'] == label].iloc[0]['ClassyFire#superclass']
        node_colors.append(mcolors.to_hex(mcolors.to_rgba(superclass_colors[related_superclass], alpha=0.6)))
    else:
        node_colors.append('#ededed')  # Default color for "Unannotated" and "All classified annotations"

# Create the Sankey plot with spaced-out nodes and consistent colors
fig = go.Figure(go.Sankey(
    node=dict(
        pad=200,
        thickness=10,
        line=dict(color='black', width=0.5),
        label=sorted_all_labels,
        x=node_x,
        y=node_y,
        color=node_colors
    ),
    link=dict(
        source=sources,
        target=targets,
        value=values,
        color='rgba(58, 158, 224, 0.3)'  #3a9ee0 blue color
    )
))

fig.update_layout(
    title=dict(
        text="Classifcation of Metabolites based on Sirius4 Annotation",
        x=0.5,  # Center the title
        xanchor='center',  # Align the title by its center
        font=dict(
            size=20,  # Adjust title font size
            color='black'  # Set the title text color (your desired color)
        )
    ),
    font=dict(size=13),  # General font size for other elements
    width=1000,
    height=600,
    margin=dict(l=10, r=225, t=50, b=10)
)

# Save the figure as PNG
fig.write_image("../Figures/metabolomics_Figures/overview/metabolomics_sankey_sirius_plot.png", format="png", width=1000, height=500, scale=2)

# Save the figure as SVG
fig.write_image("../Figures/metabolomics_Figures/overview/metabolomics_sankey_sirius_plot.svg", format="svg", width=1000, height=500)

fig.show()

/var/folders/22/yck9vwx53w1c38tvj_c0_tz00000gn/T/ipykernel_81622/3692412518.py:2: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

